# Week 4 - Q2: Regional concentration, market size or under-coverage?
**Question.** South and West generate 60.9% of net revenue; Central and North East only about 2% each. Is that market size or under-coverage?

**Approach.** Separate *how many customers* a region has from *how much each customer buys*. If revenue per customer is the same everywhere, the gap is the size of the customer base. Then check whether branch presence is linked to it.

**Definitions (same as Week 3).** Net revenue = `line_total` (ex-GST) of **Delivered** orders only. Region = the customer's region. Order value = `total_order_value` per order (not per line). Built on the corrected Week-2 merged datasets.

In [1]:
import pandas as pd, numpy as np, os
from pathlib import Path
from scipy import stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
SURF,INK,INK2,GRID,BLUE,ORANGE,AQUA,MUTED="#fcfcfb","#0b0b0b","#52514e","#e4e3df","#2a78d6","#eb6834","#1baf7a","#b9b8b2"
def find_dir(env_var, marker, rels):
    """Environment variable first, then this folder and every parent folder, using the repository layout."""
    if os.environ.get(env_var) and (Path(os.environ[env_var]) / marker).exists():
        return Path(os.environ[env_var])
    for base in [Path.cwd(), *Path.cwd().parents]:
        for rel in rels:
            if (base / rel / marker).exists():
                return base / rel
    return None
DATA_DIR = find_dir("CADETX_MERGED", "sales_orders_full.csv", ["merged", "data/merged", "week-02/reports/week_02/merged", "reports/week_02/merged"])
assert DATA_DIR is not None, "Could not find the merged datasets. Run from inside the repository, put a merged/ folder next to the notebook, or set the CADETX_MERGED environment variable to the folder."
OUT=Path("q2_outputs"); OUT.mkdir(exist_ok=True); CH=Path("q2_charts"); CH.mkdir(exist_ok=True)
checks=[]
def check(name,expected,actual,ok):
    checks.append(dict(Check_Item=name,Expected=expected,Actual=actual,Status="PASS" if ok else "FAIL")); assert ok,f"{name}: expected {expected}, got {actual}"
print("Merged datasets:",DATA_DIR.resolve())

Merged datasets: /home/claude/cs-deurghare/heavy-supplier-inventory-warehouse-analytics/week-02/reports/week_02/merged


## Part 0. Load and integrity checks

In [2]:
sales=pd.read_csv(DATA_DIR/"sales_orders_full.csv",low_memory=False,parse_dates=["order_date"])
check("sales lines",130402,len(sales),len(sales)==130402)
check("unique orders",20000,sales.so_id.nunique(),sales.so_id.nunique()==20000)
check("unique customers",500,sales.customer_id.nunique(),sales.customer_id.nunique()==500)
check("every customer is in exactly one region",1,int(sales.groupby("customer_id").customer_region.nunique().max()),sales.groupby("customer_id").customer_region.nunique().max()==1)
dl=sales[sales.order_status=="Delivered"].copy()
net=dl.line_total.sum()
check("net revenue (Delivered, ex-GST) matches Week 3, Rs 22.96B",22.96,round(net/1e9,2),round(net/1e9,2)==22.96)
allrev=sales.line_total.sum()
print(f"Delivered net revenue Rs {net/1e9:.3f}B; if cancelled orders were wrongly included: Rs {allrev/1e9:.3f}B")

Delivered net revenue Rs 22.959B; if cancelled orders were wrongly included: Rs 25.447B


## Part 1. Regional profile (Delivered orders)

In [3]:
o=dl.drop_duplicates("so_id")
g=dl.groupby("customer_region").agg(net_revenue=("line_total","sum"),customers=("customer_id","nunique"))
g["delivered_orders"]=o.groupby("customer_region").so_id.nunique()
g["avg_order_value"]=o.groupby("customer_region").total_order_value.mean()
g["revenue_share_pct"]=g.net_revenue/g.net_revenue.sum()*100
g["customer_share_pct"]=g.customers/g.customers.sum()*100
g["orders_per_customer"]=g.delivered_orders/g.customers
g["revenue_per_customer"]=g.net_revenue/g.customers
g=g.sort_values("net_revenue",ascending=False)
check("regional revenue sums to total",0,round(g.net_revenue.sum()-net,2),abs(g.net_revenue.sum()-net)<1)
check("regional customers sum to 500",500,int(g.customers.sum()),g.customers.sum()==500)
sw=g.loc[["South","West"],"revenue_share_pct"].sum()
check("South + West share is 60.9% (Week 3)",60.9,round(sw,1),round(sw,1) in (60.9,61.0) and abs(sw-60.9)<0.1)
show=g.copy(); show["net_revenue"]=show.net_revenue/1e6
show.round(2).to_csv(OUT/"q2_regional_profile.csv"); show.round(2)

,net_revenue,customers,delivered_orders,avg_order_value,revenue_share_pct,customer_share_pct,orders_per_customer,revenue_per_customer
customer_region,,,,,,,,
South,7261.74,158,5722,1269092.01,31.63,31.6,36.22,45960407.97
West,6730.78,151,5335,1261626.81,29.32,30.2,35.33,44574695.63
East,4051.95,87,3161,1281857.02,17.65,17.4,36.33,46574138.39
North,3938.17,82,3029,1300153.71,17.15,16.4,36.94,48026409.63
Central,501.84,11,406,1236065.44,2.19,2.2,36.91,45622051.82
North East,474.51,11,380,1248699.29,2.07,2.2,34.55,43136884.55


In [4]:
print(f"South + West: {sw:.2f}% of revenue and {g.loc[['South','West'],'customer_share_pct'].sum():.1f}% of customers")
print("Central + North East: %.2f%% of revenue and %.1f%% of customers"%(g.loc[["Central","North East"],"revenue_share_pct"].sum(),g.loc[["Central","North East"],"customer_share_pct"].sum()))
print("Order value per ORDER ranges Rs %.2fM to Rs %.2fM; orders per customer %.1f to %.1f; revenue per customer Rs %.1fM to Rs %.1fM"%(g.avg_order_value.min()/1e6,g.avg_order_value.max()/1e6,g.orders_per_customer.min(),g.orders_per_customer.max(),g.revenue_per_customer.min()/1e6,g.revenue_per_customer.max()/1e6))

South + West: 60.95% of revenue and 61.8% of customers
Central + North East: 4.25% of revenue and 4.4% of customers
Order value per ORDER ranges Rs 1.24M to Rs 1.30M; orders per customer 34.5 to 36.9; revenue per customer Rs 43.1M to Rs 48.0M


## Part 2. Is revenue per customer really the same in every region?
Customer-level tests (one row per customer, 500 customers).

In [5]:
pc=dl.groupby(["customer_id","customer_region"]).agg(revenue=("line_total","sum"),orders=("so_id","nunique")).reset_index()
kw_r=stats.kruskal(*[x.revenue.values for _,x in pc.groupby("customer_region")])
kw_o=stats.kruskal(*[x.orders.values for _,x in pc.groupby("customer_region")])
print(f"Revenue per customer across 6 regions: Kruskal-Wallis p = {kw_r.pvalue:.3f}; orders per customer: p = {kw_o.pvalue:.3f}")
small=pc.customer_region.isin(["Central","North East"])
rng=np.random.default_rng(42)
a=pc[small].revenue.values; b=pc[~small].revenue.values
boot=np.array([rng.choice(a,len(a)).mean()/rng.choice(b,len(b)).mean() for _ in range(5000)])
lo,hi=np.percentile(boot,[2.5,97.5]); ratio=a.mean()/b.mean()
print(f"Central + North East customers ({len(a)}) vs all others ({len(b)}): revenue per customer ratio {ratio:.2f}; bootstrap 95% interval [{lo:.2f}, {hi:.2f}]")
pd.DataFrame([["Revenue per customer, 6 regions (Kruskal-Wallis)",kw_r.statistic,kw_r.pvalue],["Orders per customer, 6 regions (Kruskal-Wallis)",kw_o.statistic,kw_o.pvalue],
  ["Revenue per customer, Central+NE vs others: ratio (bootstrap 95% low)",lo,np.nan],["Revenue per customer, Central+NE vs others: ratio",ratio,np.nan],["Revenue per customer, Central+NE vs others: ratio (bootstrap 95% high)",hi,np.nan]],columns=["test","value","p_value"]).round(4).to_csv(OUT/"q2_customer_level_tests.csv",index=False)
check("revenue per customer does not differ by region (p > 0.05)",">0.05",round(kw_r.pvalue,3),kw_r.pvalue>0.05)
check("orders per customer does not differ by region (p > 0.05)",">0.05",round(kw_o.pvalue,3),kw_o.pvalue>0.05)

Revenue per customer across 6 regions: Kruskal-Wallis p = 0.275; orders per customer: p = 0.427


Central + North East customers (22) vs all others (478): revenue per customer ratio 0.97; bootstrap 95% interval [0.88, 1.06]


## Part 2b. How strong is that conclusion?
Effect sizes, an uncertainty range for the concentration itself, and a check that a few big customers are not driving it.

In [6]:
n=len(pc); k=pc.customer_region.nunique()
eps_r=(kw_r.statistic-k+1)/(n-k); eps_o=(kw_o.statistic-k+1)/(n-k)
print(f"Effect size (epsilon-squared): revenue per customer {max(eps_r,0):.4f}, orders per customer {max(eps_o,0):.4f} (0 = no regional effect)")
rng2=np.random.default_rng(42); sw_boot=[]
for _ in range(5000):
    s=pc.iloc[rng2.integers(0,n,n)]
    sw_boot.append(s[s.customer_region.isin(["South","West"])].revenue.sum()/s.revenue.sum()*100)
sw_lo,sw_hi=np.percentile(sw_boot,[2.5,97.5])
print(f"South + West share of revenue {sw:.2f}%, bootstrap 95% interval over customers [{sw_lo:.1f}%, {sw_hi:.1f}%]")
top=pc.sort_values("revenue",ascending=False)
top5=top.head(5).revenue.sum()/pc.revenue.sum()*100; top_one=top.iloc[0].revenue/pc.revenue.sum()*100
print(f"Largest single customer is {top_one:.2f}% of revenue; top 5 customers are {top5:.2f}%")
ex5=pc.drop(top.head(5).index); sw_ex=ex5[ex5.customer_region.isin(["South","West"])].revenue.sum()/ex5.revenue.sum()*100
print(f"South + West share without the top 5 customers: {sw_ex:.2f}% (vs {sw:.2f}%)")
print(f"Interval for the Central + North East ratio is 0.88 to 1.06, so a gap larger than about 12% below other regions is ruled out; with only 22 customers a smaller gap cannot be excluded.")
pd.DataFrame([["epsilon-squared, revenue per customer",max(eps_r,0)],["epsilon-squared, orders per customer",max(eps_o,0)],["South+West revenue share (%)",sw],["South+West share bootstrap low (%)",sw_lo],["South+West share bootstrap high (%)",sw_hi],["largest customer share of revenue (%)",top_one],["top 5 customers share of revenue (%)",top5],["South+West share without top 5 (%)",sw_ex]],columns=["measure","value"]).round(4).to_csv(OUT/"q2_robustness.csv",index=False)
check("regional effect on revenue per customer is negligible (epsilon-squared < 0.01)","<0.01",round(max(eps_r,0),4),max(eps_r,0)<0.01)
check("South + West share stays above 55% across the bootstrap interval",">55",round(sw_lo,1),sw_lo>55)
check("no single customer drives the concentration (largest < 1% of revenue)","<1",round(top_one,2),top_one<1)
check("removing the top 5 customers moves the South + West share by less than those customers' own revenue share (%)","<%.2f"%top5,round(abs(sw_ex-sw),2),abs(sw_ex-sw)<top5)

Effect size (epsilon-squared): revenue per customer 0.0027, orders per customer 0.0000 (0 = no regional effect)


South + West share of revenue 60.95%, bootstrap 95% interval over customers [56.5%, 65.3%]
Largest single customer is 0.37% of revenue; top 5 customers are 1.68%
South + West share without the top 5 customers: 61.99% (vs 60.95%)
Interval for the Central + North East ratio is 0.88 to 1.06, so a gap larger than about 12% below other regions is ruled out; with only 22 customers a smaller gap cannot be excluded.


## Part 3. Branch presence versus where customers are

In [7]:
br=dl.groupby("branch_region").agg(branches=("branch_id","nunique"),net_revenue=("line_total","sum"))
print(br.assign(net_revenue=br.net_revenue/1e9).round(2).rename(columns={"net_revenue":"net_revenue_B"}).to_string())
ct=pd.crosstab(o.customer_region,o.branch_region); ct.to_csv(OUT/"q2_customer_region_vs_branch_region.csv")
has_branch=set(br.index)
check("Central and North East have no branch",True,sorted(set(["Central","North East"])-has_branch),set(["Central","North East"])&has_branch==set())
print("\nShare of each customer region's orders handled by a branch in the SAME region (%):")
same=pd.Series({r:(ct.loc[r,r]/ct.loc[r].sum()*100 if r in ct.columns else 0.0) for r in ct.index}); print(same.round(1).to_string())
chi2,pchi,_,_=stats.chi2_contingency(ct.values); cv=np.sqrt(chi2/(ct.values.sum()*(min(ct.shape)-1)))
print(f"\nCustomer region vs branch region (orders): chi-square p {('< 1e-300' if pchi==0 else '= %.3g'%pchi)}, Cramer's V = {cv:.2f} (orders mostly go to a branch in the customer's own region, where one exists)")
check("customer region and branch region are strongly linked (Cramer's V > 0.5)",">0.5",round(cv,2),cv>0.5)

               branches  net_revenue_B
branch_region                         
East                  1           4.34
North                 1           3.50
South                 2           7.76
West                  2           7.36

Share of each customer region's orders handled by a branch in the SAME region (%):
Central        0.0
East          76.5
North         73.3
North East     0.0
South         87.7
West          86.5

Customer region vs branch region (orders): chi-square p < 1e-300, Cramer's V = 0.74 (orders mostly go to a branch in the customer's own region, where one exists)


## Chart

In [8]:
gg=g.sort_values("revenue_share_pct"); y=np.arange(len(gg)); h=0.36
fig,ax=plt.subplots(figsize=(8.5,4.6),facecolor=SURF); ax.set_facecolor(SURF)
ax.barh(y+h/2,gg.revenue_share_pct,height=h,color=BLUE,label="Share of net revenue")
ax.barh(y-h/2,gg.customer_share_pct,height=h,color=ORANGE,label="Share of customers")
for yi,(r,c) in enumerate(zip(gg.revenue_share_pct,gg.customer_share_pct)):
    ax.text(r+0.4,yi+h/2,f"{r:.1f}%",va="center",fontsize=8.5,color=INK2); ax.text(c+0.4,yi-h/2,f"{c:.1f}%",va="center",fontsize=8.5,color=INK2)
ax.set_yticks(y); ax.set_yticklabels(gg.index,color=INK2); ax.set_xlabel("Percent of total",color=INK2); ax.grid(axis="x",color=GRID,lw=.6); ax.set_axisbelow(True)
[s.set_visible(False) for k,s in ax.spines.items() if k in("top","right")]
ax.set_title("Each region's share of revenue matches its share of customers",loc="left",color=INK,fontsize=12,fontweight="bold"); ax.legend(frameon=False,loc="lower right")
plt.tight_layout(); plt.savefig(CH/"q2_01_revenue_vs_customer_share.png",dpi=150,facecolor=SURF); plt.close()

## Q2 answer

| Finding | Evidence |
|---|---|
| **The gap is the size of the customer base, not customer behaviour.** South + West hold about 61% of revenue and about 62% of customers. | Part 1 |
| **A customer is worth about the same everywhere.** About Rs 43-48M per customer over six years, 35-37 orders each, about Rs 1.24-1.30M per order. No regional difference (Kruskal-Wallis p = 0.28 for revenue and 0.43 for orders). | Parts 1-2 |
| **Central and North East have only 11 customers each** (of 500), so they are about 2% of revenue each. Their customers buy about the same as others (revenue per customer ratio 0.97, 95% interval 0.88 to 1.06, from only 22 customers). | Part 2 |
| **The result is robust.** Regional effect sizes are negligible (epsilon-squared below 0.01); the South + West share has a tight bootstrap interval; no single customer drives it; removing the top 5 customers (1.7% of revenue) moves the South + West share by about 1 point (60.9% to 62.0%). With only 22 customers a Central + North East gap smaller than about 12% cannot be excluded. | Part 2b |
| **Central and North East have no branch.** Their orders are handled from other regions' branches. | Part 3 |
| **What the data cannot say:** whether Central and North East are *small markets* or *unserved markets*. There is no market-potential field. | |
| **Recommendation:** present the finding as "concentration = customer-base size"; test customer acquisition in Central and North East before adding branches. If new customers behaved like existing ones (an assumption, not tested), each would be worth about Rs 45M over six years. | |

## Validation report

In [9]:
v=pd.DataFrame(checks); v.to_csv("validation_report_week04_q2.csv",index=False); print(v.to_string(index=False)); assert (v.Status=="PASS").all()

                                                                                                   Check_Item Expected                Actual Status
                                                                                                  sales lines   130402                130402   PASS
                                                                                                unique orders    20000                 20000   PASS
                                                                                             unique customers      500                   500   PASS
                                                                      every customer is in exactly one region        1                     1   PASS
                                                    net revenue (Delivered, ex-GST) matches Week 3, Rs 22.96B    22.96                 22.96   PASS
                                                                               regional revenue sums to total   